In [21]:
import pandas as pd
import spacy
from collections import defaultdict 
from spacy import displacy
import random
from spacy.training import Example
from spacy.pipeline import EntityRecognizer
from typing import List, Tuple
from sentence_transformers import SentenceTransformer
import nltk
from nltk.corpus import stopwords
import string
import re
import numpy as np
from tqdm import tqdm

# 1. Загрузка датасета

In [22]:
df = pd.read_json("../../data/data.txt", lines=True)
final_df = df.copy()

# 2. Используем стандартную NER-модель

In [23]:
# Загружаем модель для русского языка
# если она не найдена, то установим командой: python -m spacy download ru_core_news_lg
# также попробовать deeppavlovru_core_news_lg - https://chat.deepseek.com/a/chat/s/3d0a9423-ea38-4e29-8dc5-63b5ec94b45a
nlp = spacy.load("ru_core_news_lg")

In [24]:
def extract_entities_from_texts(texts):
    results = []
    for doc in nlp.pipe(texts, batch_size=50, n_process=-1):  # n_process=-1 использует все ядра
        entities = defaultdict(set)
        for ent in doc.ents:
            entities[ent.label_].add(ent.text)
        results.append(dict(entities))
    return results

final_df['entities'] = extract_entities_from_texts(final_df['text'].str.lower().tolist())

In [25]:
final_df



,text,tags,schema_name,table_name,entities
0,"1 1 1 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47923,"{'LOC': {'москвы', 'рф'}, 'ORG': {'минприроды'}}"
1,"1 1 1 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_58710,"{'LOC': {'москвы', 'российской федерации'}}"
2,"2 2 2 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_58710,{'LOC': {'москвы'}}
3,"3 3 3 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_58710,{'LOC': {'москвы'}}
4,"4 4 4 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_58710,{'LOC': {'москвы'}}
...,...,...,...,...,...
18419,996 996 982 20231117_110117.jpg 2023-11-17 00:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876,{}
18420,997 997 983 20231121_100442.jpg 2023-11-21 00:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876,{'PER': {'пип битцевский'}}
18421,998 998 984 20231121_095847.jpg 2023-11-21 00:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876,{'PER': {'пип битцевский'}}
18422,99 99 101 117 4.jpg 2023-11-16 00:00:00+00 13:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876,{}


In [26]:
# визуализация
# text = final_df[5:6].reset_index()['text'].str.lower()[0]
text = final_df[100:101].reset_index()['text'][0]
doc = nlp(text)
displacy.render(doc, style="ent", jupyter=True)

# 3. Добавим labels и дообучим существующую NER-модель

In [27]:
# СЗАО - Северо-Западный административный округ
# САО - Северный административный округ
# СВАО - Северо-Восточный административный округ
# ЗАО - Западный административный округ
# ЦАО - Центральный административный округ
# ВАО - Восточный административный округ
# ЮЗАО - Юго-Западный административный округ
# ЮАО - Южный административный округ
# ЮВАО - Юго-Восточный административный округ
# ЗелАО - Зеленоградский административный округ
# ТиНАО - Троицкий и Новомосковский административные округа
# НАО - Новомосковский административный округ
# ТАО - Троицкий административный округ

In [28]:
result_df = df.copy()
moscow_districts = ['СЗАО', 'САО', 'СВАО', 'ЗАО', 'ЦАО', 'ВАО', 'ЮЗАО', 'ЮАО', 'ЮВАО', 'ЗелАО', 'ТиНАО', 'НАО', 'ТАО']
# Создаем регулярное выражение для поиска любого из значений
pattern = '|'.join(moscow_districts)
result_df = df[df['text'].str.contains(pattern, case=False, na=False)]

In [29]:
result_df

,text,tags,schema_name,table_name
72,"100 101 64 ООПТ регионального значения ""Памятн...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_54935
75,"102 103 3 ООПТ регионального значения ""Памятни...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_54935
118,"141 142 120 ООПТ регионального значения ""Фауни...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_54935
144,"31 31 138 ООПТ регионального значения ""Памятни...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_54935
215,"96 97 38 ООПТ регионального значения ""Природно...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_54935
...,...,...,...,...
18419,996 996 982 20231117_110117.jpg 2023-11-17 00:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876
18420,997 997 983 20231121_100442.jpg 2023-11-21 00:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876
18421,998 998 984 20231121_095847.jpg 2023-11-21 00:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876
18422,99 99 101 117 4.jpg 2023-11-16 00:00:00+00 13:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876


In [30]:
def find_all_positions(text, substring):
    """Находит все позиции вхождения подстроки в тексте"""
    positions = []
    start = 0
    while True:
        pos = text.find(substring, start)
        if pos == -1:
            break
        positions.append([pos, pos + len(substring)])
        start = pos + len(substring) # ищем следующее вхождение
    return positions
# Пример использования
text = "СЗАО и ВАО - это округа Москвы. СЗАО находится на северо-западе. СЗАО" 
substring = "СЗАО"
positions = find_all_positions(text, substring)
print(f"Подстрока '{substring}' найдена на позициях: {positions}")

Подстрока 'СЗАО' найдена на позициях: [[0, 4], [32, 36], [65, 69]]


In [31]:
def get_examples(df, entities_dict, nlp):
    all_annotated_examples = []
    texts = df['text'].tolist()
    for text in texts:
        example_for_text = []
        entities_for_text = []
        for entity in entities_dict:
            for value in entities_dict[entity]:
                # nonlocal positions
                positions = find_all_positions(text, value)
                for position in positions:
                    position.append(entity)
                    entities_for_text.append(tuple(position))
        
        doc = nlp(text)
        for ent in doc.ents:
            entities_for_text.append((ent.start_char, ent.end_char, ent.label_))
            
        if len(entities_for_text) > 0:
            example_for_text.insert(0, text)
            example_for_text.append({'entities': entities_for_text})
            all_annotated_examples.append(example_for_text)
    return all_annotated_examples

entities_values = {"DISTRICT":moscow_districts}
annotated_examples = get_examples(result_df[:100], entities_values, nlp)
print(annotated_examples)
        

[['100 101 64 ООПТ регионального значения "Памятник природы "Долина реки Чермянки от пр. Дежнева до устья" ППМ № 2119-ПП от 18.09.2024,ППМ № 1496-ПП от 11.09.2020 Памятники природы Утвержден ГБУ г. Москвы "Автомобильные дороги СВАО" https://docs7.online-sps.ru/cgi/online.cgi?from=228884-0&req=doc&rnd=JMzqYA&base=MLAW&n=246251#mUvXRTUQQIa3BB3w https://www.mos.ru/upload/content/files/020KDPPDolinarekiChermyankiotprDejnevadoystya(2).docx 15.55 15.5477 не совпадает с зонами режимов ООПТ (по координатам в ППМ - такая же геометрия) Иль С.А.: Проверен Мукаяров Е.А.: Внесено в соответствии с ППМ 2119 от 18.09.24', {'entities': [(224, 228, 'DISTRICT'), (225, 228, 'DISTRICT'), (70, 78, 'LOC'), (86, 93, 'PER'), (104, 117, 'ORG'), (195, 201, 'LOC'), (529, 537, 'PER'), (539, 561, 'PER'), (588, 591, 'ORG')]}], ['102 103 3 ООПТ регионального значения "Памятник природы "Пойма реки Городни от Братеевской ул. до реки Москвы" ППМ № 2406-ПП от 23.10.2024,ППМ № 1540-ПП от 16.09.2020 Памятники природы Утвер

In [32]:
def resolve_overlapping_entities(training_sample):
    """
    Удаляет все пересекающиеся сущности, оставляя только непересекающиеся.
    При конфликте выбирает сущность с наибольшей длиной.
    """
    text, annotations = training_sample
    entities = annotations.get('entities', [])
    
    if not entities:
        return training_sample
    
    # Удаляем дубликаты
    unique_entities = list(set(tuple(entity) for entity in entities))
    
    # Сортируем по длине (от самой длинной к самой короткой)
    sorted_by_length = sorted(unique_entities, 
                             key=lambda x: (x[1] - x[0]), 
                             reverse=True)
    
    result_entities = []
    used_positions = set()
    
    for entity in sorted_by_length:
        start, end, label = entity
        entity_positions = set(range(start, end))
        
        # Проверяем, пересекается ли с уже добавленными сущностями
        if not entity_positions.intersection(used_positions):
            result_entities.append(entity)
            used_positions.update(entity_positions)
    
    # Сортируем по начальной позиции
    result_entities.sort(key=lambda x: x[0])
    
    return [text, {'entities': result_entities}]

In [33]:
resolved_annotated_examples = []
for example in annotated_examples:
    resolved_example = resolve_overlapping_entities(example)
    resolved_annotated_examples.append(resolved_example)
print(resolved_annotated_examples)

[['100 101 64 ООПТ регионального значения "Памятник природы "Долина реки Чермянки от пр. Дежнева до устья" ППМ № 2119-ПП от 18.09.2024,ППМ № 1496-ПП от 11.09.2020 Памятники природы Утвержден ГБУ г. Москвы "Автомобильные дороги СВАО" https://docs7.online-sps.ru/cgi/online.cgi?from=228884-0&req=doc&rnd=JMzqYA&base=MLAW&n=246251#mUvXRTUQQIa3BB3w https://www.mos.ru/upload/content/files/020KDPPDolinarekiChermyankiotprDejnevadoystya(2).docx 15.55 15.5477 не совпадает с зонами режимов ООПТ (по координатам в ППМ - такая же геометрия) Иль С.А.: Проверен Мукаяров Е.А.: Внесено в соответствии с ППМ 2119 от 18.09.24', {'entities': [(70, 78, 'LOC'), (86, 93, 'PER'), (104, 117, 'ORG'), (195, 201, 'LOC'), (224, 228, 'DISTRICT'), (529, 537, 'PER'), (539, 561, 'PER'), (588, 591, 'ORG')]}], ['102 103 3 ООПТ регионального значения "Памятник природы "Пойма реки Городни от Братеевской ул. до реки Москвы" ППМ № 2406-ПП от 23.10.2024,ППМ № 1540-ПП от 16.09.2020 Памятники природы Утвержден ГБУ г. Москвы "Авто

In [34]:
# Получаем NER компонент
ner = nlp.get_pipe("ner")

In [35]:
# добавляем новые labels
for label in entities_values:
    if label not in ner.labels:
        ner.add_label(label)
        print(f"Добавлен новый label: {label}")

Добавлен новый label: DISTRICT


In [36]:
# отключаем все остальные pipe
other_pipes = [pipe for pipe in nlp.pipe_names if pipe != "ner"]
with nlp.disable_pipes(*other_pipes):
    optimizer = nlp.resume_training()

In [37]:
# 
iteration_number = 5
for iteration in range(iteration_number):
    random.shuffle(resolved_annotated_examples)
    losses = {}
    for example in resolved_annotated_examples:
        text = example[0]
        annotations = example[1]
        doc = nlp.make_doc(text)
        train_example = Example.from_dict(doc, annotations)
        nlp.update([train_example], sgd=optimizer, losses=losses, drop=0.4)
    if iteration % 5 == 0:
        print(f"Iteration {iteration}, Loss: {losses['ner']}")


Iteration 0, Loss: 3221.941162109375


KeyboardInterrupt: 

In [35]:
# сохранение модели
ner = nlp.get_pipe('ner')
# ner.to_disk('../files')

In [36]:
# загрузка модели
# nlp = spacy.load('ru_core_news_lg', disable=['ner'])
# ner = EntityRecognizer(nlp.vocab)
# ner.from_disk('/usr/to/ner')
# nlp.add_pipe(ner, "custom_ner")
# 
# print(nlp.meta['pipeline'])

In [37]:
# работа обновленной модели
sample_text = ('ЦАО Пресненский (ЦАО) 159 158 Постановление Правительства Москвы от 01.10.2020 № 1642-ПП \"Об образовании особо охраняемых природных территорий регионального значения - памятников природы в городе Москве\" аккумулятивный полого-волнистый моренный рельеф Префектура ЦАО, ГБУ \"Автомобильные Дороги ЦАО\" '
               'Лосиный остров СВАО САО')

doc = nlp(sample_text)
entities = []
for ent in doc.ents:
    entities.append({
        'text': ent.text,
        'label': ent.label_,
        'start': ent.start_char,
        'end': ent.end_char
    })
for i, ent in enumerate(entities, 1):
    print(f"{i}. {ent['text']} -> {ent['label']} ({ent['start']}-{ent['end']})")
displacy.render(doc, style="ent", jupyter=True)

1. ЦАО Пресненский -> PER (0-15)
2. ЦАО -> DISTRICT (17-20)
3. Москвы -> LOC (58-64)
4. Москве -> LOC (196-202)
5. Префектура -> LOC (252-262)
6. ЦАО -> DISTRICT (263-266)
7. ГБУ "Автомобильные Дороги ЦАО" Лосиный остров -> ORG (268-313)
8. СВАО -> LOC (314-318)
9. САО -> DISTRICT (319-322)
